# Solution: Planning a telescope observation — Markov Chain weather forecast

We simulate a Markov Chain for the weather (clear vs cloudy) with transition probabilities:

- P(clear | cloudy) = 0.5
- P(cloudy | cloudy) = 0.5
- P(cloudy | clear) = 0.1
- P(clear | clear) = 0.9

Expected equilibrium: P(clear) ≈ 0.83, P(cloudy) ≈ 0.17

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format='retina'

## 1. Define transition probabilities and run the Markov Chain

In [ ]:
# Transition probabilities
# State: 0 = cloudy, 1 = clear
p_clear_given_cloudy = 0.5
p_clear_given_clear  = 0.9

# Number of days to simulate
N = 50000

# Start on a cloudy day (state = 0)
np.random.seed(42)
chain = np.zeros(N, dtype=int)
chain[0] = 0  # cloudy

for i in range(1, N):
    r = np.random.random()
    if chain[i-1] == 0:  # today is cloudy
        chain[i] = 1 if r < p_clear_given_cloudy else 0
    else:                # today is clear
        chain[i] = 1 if r < p_clear_given_clear else 0

print(f"Total days simulated: {N}")
print(f"Clear days: {np.sum(chain)}")
print(f"Cloudy days: {N - np.sum(chain)}")
print(f"Fraction of clear days: {np.mean(chain):.4f}")
print(f"Expected equilibrium:    0.8333")

## 2. Trace plot: cumulative fraction of clear days

In [ ]:
# Compute cumulative fraction of clear days
cumulative_clear = np.cumsum(chain)
days = np.arange(1, N + 1)
fraction_clear = cumulative_clear / days

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(days, fraction_clear, lw=0.5)
ax.axhline(5/6, color='r', ls='--', label=r'Equilibrium $p(\mathrm{clear})=5/6\approx0.833$')
ax.set_xlabel('Number of days')
ax.set_ylabel(r'Cumulative $p(\mathrm{clear})$')
ax.set_title('Trace plot')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Histogram of cumulative fraction (after burn-in)

In [ ]:
# Discard burn-in: first 500 steps
burn_in = 500
chain_post = chain[burn_in:]

# Recompute running fraction after burn-in
cumulative_post = np.cumsum(chain_post)
days_post = np.arange(1, len(chain_post) + 1)
fraction_post = cumulative_post / days_post

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(fraction_post, bins=80, density=True, alpha=0.7, label='After burn-in')
ax.axvline(5/6, color='r', ls='--', lw=2, label=r'Equilibrium $p(\mathrm{clear})=5/6$')
ax.set_xlabel(r'Cumulative $p(\mathrm{clear})$')
ax.set_ylabel('Density')
ax.set_title('Distribution of running clear-day fraction')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Summary statistics

In [ ]:
p_clear_estimate = np.mean(chain_post)
p_clear_std = np.std(chain_post) / np.sqrt(len(chain_post))

print(f"Estimated p(clear) = {p_clear_estimate:.5f} ± {p_clear_std:.5f}")
print(f"Theoretical value  = {5/6:.5f}")
print(f"")
print(f"Estimated p(cloudy) = {1 - p_clear_estimate:.5f}")
print(f"Theoretical value   = {1/6:.5f}")

## 5. Experimenting with burn-in

Here we try different burn-in lengths and see how the estimate of p(clear) changes.

In [ ]:
burn_in_values = [0, 10, 50, 100, 500, 1000, 5000]

fig, axes = plt.subplots(len(burn_in_values), 1, figsize=(10, 3*len(burn_in_values)))

for ax, bi in zip(axes, burn_in_values):
    chain_bi = chain[bi:]
    cum = np.cumsum(chain_bi) / np.arange(1, len(chain_bi)+1)
    ax.plot(np.arange(bi, N), cum, lw=0.5)
    ax.axhline(5/6, color='r', ls='--', lw=1)
    ax.set_ylabel(r'$p(\mathrm{clear})$')
    ax.set_title(f'Burn-in = {bi}')
    estimate = np.mean(chain_bi)
    ax.text(0.98, 0.05, f'Mean = {estimate:.4f}', transform=ax.transAxes, ha='right', fontsize=10)

axes[-1].set_xlabel('Day')
plt.tight_layout()
plt.show()

In [ ]:
# Print summary table
print(f"{'Burn-in':>10s} | {'p(clear) estimate':>18s} | {'Error vs theory':>16s}")
print('-' * 52)
for bi in burn_in_values:
    est = np.mean(chain[bi:])
    err = abs(est - 5/6)
    print(f"{bi:>10d} | {est:>18.5f} | {err:>16.6f}")